# 04 - Model Development

Comparing two candidates: **K-Means + Random Forest** (hybrid, cluster-distance features feeding RF) and **XGBoost** (with class-imbalance weighting). Best one is selected and saved for evaluation/XAI.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

DATA_PATH = Path("../data/processed")
MODEL_PATH = Path("../outputs/models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_PATH / "train_processed.csv")
test = pd.read_csv(DATA_PATH / "test_processed.csv")

X_train = train.drop(columns="Outcome")
y_train = train["Outcome"]
X_test = test.drop(columns="Outcome")
y_test = test["Outcome"]

print(X_train.shape, X_test.shape)


(614, 8) (154, 8)


**Model A: K-Means + Random Forest**

Find best k via silhouette score (fit on train only), then use each point's distance-to-centroid as extra engineered features for a class-weighted Random Forest.

In [2]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

k_values = list(range(2, 10))
silhouette_scores = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_train)
    silhouette_scores.append(silhouette_score(X_train, labels))

best_k = k_values[np.argmax(silhouette_scores)]
print("Silhouette scores:", dict(zip(k_values, np.round(silhouette_scores, 4))))
print("Best k =", best_k)


Silhouette scores: {2: np.float64(0.2161), 3: np.float64(0.1763), 4: np.float64(0.1656), 5: np.float64(0.1545), 6: np.float64(0.1576), 7: np.float64(0.1573), 8: np.float64(0.1455), 9: np.float64(0.1555)}
Best k = 2


In [3]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans.fit(X_train)

dist_cols = [f"Cluster_Dist_{i+1}" for i in range(best_k)]

train_dist = pd.DataFrame(kmeans.transform(X_train), columns=dist_cols)
test_dist = pd.DataFrame(kmeans.transform(X_test), columns=dist_cols)

X_train_km = pd.concat([X_train.reset_index(drop=True), train_dist], axis=1)
X_test_km = pd.concat([X_test.reset_index(drop=True), test_dist], axis=1)

print(X_train_km.shape, X_test_km.shape)


(614, 10) (154, 10)


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

kmeans_rf = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", random_state=42
)

acc_scores_rf = cross_val_score(kmeans_rf, X_train_km, y_train, cv=cv, scoring="accuracy")
auc_scores_rf = cross_val_score(kmeans_rf, X_train_km, y_train, cv=cv, scoring="roc_auc")

print(f"K-Means + RF -- CV Accuracy: {acc_scores_rf.mean():.4f} (+/-{acc_scores_rf.std():.4f})")
print(f"K-Means + RF -- CV ROC AUC : {auc_scores_rf.mean():.4f}")


K-Means + RF -- CV Accuracy: 0.7362 (+/-0.0194)
K-Means + RF -- CV ROC AUC : 0.8220


**Model B: XGBoost (class-imbalance weighted, on original engineered features)**

In [5]:
from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_base = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=42
)

acc_scores_xgb = cross_val_score(xgb_base, X_train, y_train, cv=cv, scoring="accuracy")
auc_scores_xgb = cross_val_score(xgb_base, X_train, y_train, cv=cv, scoring="roc_auc")

print(f"XGBoost -- CV Accuracy: {acc_scores_xgb.mean():.4f} (+/-{acc_scores_xgb.std():.4f})")
print(f"XGBoost -- CV ROC AUC : {auc_scores_xgb.mean():.4f}")


XGBoost -- CV Accuracy: 0.7557 (+/-0.0334)
XGBoost -- CV ROC AUC : 0.8143


**Side-by-side comparison**

In [6]:
results_df = pd.DataFrame([
    {"Model": "K-Means + Random Forest", "CV Accuracy": acc_scores_rf.mean(), "CV Accuracy Std": acc_scores_rf.std(), "CV ROC AUC": auc_scores_rf.mean()},
    {"Model": "XGBoost",                 "CV Accuracy": acc_scores_xgb.mean(), "CV Accuracy Std": acc_scores_xgb.std(), "CV ROC AUC": auc_scores_xgb.mean()},
]).sort_values("CV ROC AUC", ascending=False)

results_df


,Model,CV Accuracy,CV Accuracy Std,CV ROC AUC
0,K-Means + Random Forest,0.736172,0.019424,0.822002
1,XGBoost,0.755684,0.033434,0.814302


**Hyperparameter tuning — XGBoost**

In [7]:
from sklearn.model_selection import GridSearchCV

xgb_param_grid = {
    "n_estimators": [200, 300, 400],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

xgb_grid = GridSearchCV(
    estimator=XGBClassifier(
        random_state=42, eval_metric="logloss", scale_pos_weight=scale_pos_weight
    ),
    param_grid=xgb_param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
)
xgb_grid.fit(X_train, y_train)

print("Best params:", xgb_grid.best_params_)
print("Best CV ROC AUC:", xgb_grid.best_score_)

best_xgb = xgb_grid.best_estimator_


Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
Best CV ROC AUC: 0.8419601328903654


**Hyperparameter tuning — K-Means + Random Forest**

In [8]:
rf_param_grid = {
    "n_estimators": [200, 300, 400],
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(class_weight="balanced", random_state=42),
    param_grid=rf_param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
)
rf_grid.fit(X_train_km, y_train)

print("Best params:", rf_grid.best_params_)
print("Best CV ROC AUC:", rf_grid.best_score_)

best_kmeans_rf = rf_grid.best_estimator_


Best params: {'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 300}
Best CV ROC AUC: 0.8360908084163899


**Fit tuned models on full train set, quick holdout check (full evaluation happens in notebook 05)**

In [9]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

best_xgb.fit(X_train, y_train)
y_pred_xgb = best_xgb.predict(X_test)
y_prob_xgb = best_xgb.predict_proba(X_test)[:, 1]

best_kmeans_rf.fit(X_train_km, y_train)
y_pred_km_rf = best_kmeans_rf.predict(X_test_km)
y_prob_km_rf = best_kmeans_rf.predict_proba(X_test_km)[:, 1]

print("XGBoost -- Test Accuracy:", accuracy_score(y_test, y_pred_xgb), " ROC AUC:", roc_auc_score(y_test, y_prob_xgb))
print(classification_report(y_test, y_pred_xgb))

print("K-Means + RF -- Test Accuracy:", accuracy_score(y_test, y_pred_km_rf), " ROC AUC:", roc_auc_score(y_test, y_prob_km_rf))
print(classification_report(y_test, y_pred_km_rf))


XGBoost -- Test Accuracy: 0.7272727272727273  ROC AUC: 0.8175
              precision    recall  f1-score   support

           0       0.84      0.72      0.77       100
           1       0.59      0.74      0.66        54

    accuracy                           0.73       154
   macro avg       0.71      0.73      0.71       154
weighted avg       0.75      0.73      0.73       154

K-Means + RF -- Test Accuracy: 0.7142857142857143  ROC AUC: 0.8107407407407408
              precision    recall  f1-score   support

           0       0.84      0.69      0.76       100
           1       0.57      0.76      0.65        54

    accuracy                           0.71       154
   macro avg       0.71      0.72      0.70       154
weighted avg       0.75      0.71      0.72       154



**Select the better model and save everything**

In [10]:
selected_name = "XGBoost" if roc_auc_score(y_test, y_prob_xgb) >= roc_auc_score(y_test, y_prob_km_rf) else "K-Means + Random Forest"
print("Selected model:", selected_name)

joblib.dump(best_xgb, MODEL_PATH / "xgboost_final_model.pkl")
joblib.dump(best_kmeans_rf, MODEL_PATH / "kmeans_rf_final_model.pkl")
joblib.dump(kmeans, MODEL_PATH / "kmeans_clusterer.pkl")
joblib.dump({"selected_model": selected_name}, MODEL_PATH / "selection_metadata.pkl")

# Save the K-Means-augmented feature sets too, since notebook 05/06 need them if K-Means+RF is selected
X_train_km.to_csv(DATA_PATH / "X_train_kmeans_features.csv", index=False)
X_test_km.to_csv(DATA_PATH / "X_test_kmeans_features.csv", index=False)

print("Saved both models, the clusterer, and the K-Means feature sets.")


Selected model: XGBoost
Saved both models, the clusterer, and the K-Means feature sets.
